## 1. 라이브러리와 실험 상수

In [ ]:
from __future__ import annotations
import argparse
import csv
import json
from pathlib import Path
import numpy as np

TARGET_LANGS = ["de", "fr", "el", "fi", "pl", "bg", "mt"]
CATEGORIES = [
    "legal_terminology", "long_document", "multiple_labels",
    "proper_nouns_institutions", "numbers_dates_articles", "translation_expression"
]

## 2. 입출력 및 F1 계산 함수

In [ ]:
def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def write_csv(path, rows, fieldnames):
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)

def doc_counts(y, prob, threshold):
    actual = y.astype(bool)
    pred = prob >= threshold
    return np.stack(((actual & pred).sum(axis=1),
                     ((~actual) & pred).sum(axis=1),
                     (actual & (~pred)).sum(axis=1)), axis=1).astype(np.float32)

def micro_f1(s):
    tp, fp, fn = (s[..., i].astype(np.float64) for i in range(3))
    d = 2 * tp + fp + fn
    return np.divide(2 * tp, d, out=np.zeros_like(d), where=d > 0)

def doc_f1(y, prob, threshold):
    c = doc_counts(y, prob, threshold)
    return micro_f1(c)

def pred_labels(p, threshold):
    return [int(i) for i in np.flatnonzero(p >= threshold)]

def true_labels(y):
    return [int(i) for i in np.flatnonzero(y)]

## 3. CELEX 기준 Paired Bootstrap CI

In [ ]:
def bootstrap_rows(y, preds, metas, n_boot, seed, batch_size=100):
    """One exact paired CELEX-resampling scheme shared by all comparisons."""
    all_models = {}
    for cond in ("standard", "diagnostic", "direct"):
        for lang, item in metas[cond]["results"].items():
            p = preds[cond][lang]
            assert p.shape == y.shape, (cond, lang, p.shape)
            assert p.dtype == np.float32
            assert np.isfinite(p).all()
            assert p.min() >= 0 and p.max() <= 1
            c = doc_counts(y, p, float(item["threshold"]))
            got = float(micro_f1(c.sum(axis=0)))
            target = float(item["metrics"]["micro_f1"])
            if abs(got - target) > 1e-7:
                raise AssertionError((cond, lang, got, target))
            all_models[(cond, lang)] = c

    names = list(all_models)
    col = {name: i for i, name in enumerate(names)}
    # Matrix of document-wise (TP, FP, FN) for all runs, maintaining common CELEX alignment.
    features = np.concatenate([all_models[k] for k in names], axis=1)  # (5000, 3 * models)
    observed_sums = features.sum(axis=0).reshape(len(names), 3)
    observed_f1s = micro_f1(observed_sums)
    results = np.empty((n_boot, len(names)), dtype=np.float64)
    rng = np.random.default_rng(seed)
    n = len(y)

    for start in range(0, n_boot, batch_size):
        b = min(batch_size, n_boot - start)
        idx = rng.integers(0, n, size=(b, n), dtype=np.int32)
        # W[i, j] = number of occurrences of CELEX document j in bootstrap sample i.
        flattened = idx + n * np.arange(b, dtype=np.int32)[:, None]
        W = np.bincount(flattened.ravel(), minlength=b * n).reshape(b, n).astype(np.float32)
        totals = (W @ features).reshape(b, len(names), 3)
        results[start:start+b] = micro_f1(totals)

    comparisons = []
    for lang in TARGET_LANGS:
        comparisons.extend([
            ("transfer_gap", lang, ("direct", lang), ("standard", lang)),
            ("residual_gap", lang, ("direct", lang), ("diagnostic", lang)),
            ("threshold_effect", lang, ("diagnostic", lang), ("standard", lang)),
            ("rq1_en_minus_direct", lang, ("direct", "en"), ("direct", lang)),
        ])
    rows = []
    for name, lang, left, right in comparisons:
        ai, bi = col[left], col[right]
        dif = (results[:, ai] - results[:, bi]) * 100
        lower, upper = np.quantile(dif, [0.025, 0.975])
        observed = (observed_f1s[ai] - observed_f1s[bi]) * 100
        rows.append({
            "analysis": name,
            "lang": lang.upper(),
            "left_condition": left[0],
            "left_lang": left[1].upper(),
            "right_condition": right[0],
            "right_lang": right[1].upper(),
            "observed_gap_pp": round(float(observed), 6),
            "ci_95_low_pp": round(float(lower), 6),
            "ci_95_high_pp": round(float(upper), 6),
            "ci_excludes_0": bool(lower > 0 or upper < 0),
            "n_docs": n,
            "n_boot": n_boot,
            "seed": seed,
            "ci_method": "percentile",
            "paired_by": "ordered CELEX ID",
        })
    return rows, len(names)

## 4. Flip Case 선정 및 D/E 코딩 시트 생성

In [ ]:
def make_flips(y, ids, std, metas, bootstrap, seed):
    gaprows = [r for r in bootstrap if r["analysis"] == "transfer_gap"]
    high = max(gaprows, key=lambda x: x["observed_gap_pp"])["lang"].lower()
    low = min(gaprows, key=lambda x: x["observed_gap_pp"])["lang"].lower()
    targets = [(high, "largest_transfer_gap"), (low, "smallest_transfer_gap")]
    f1_en = doc_f1(y, std["en"], float(metas["standard"]["results"]["en"]["threshold"]))
    all_candidates, selected, coding_d, coding_e = [], [], [], []
    case_stats = []
    for lang, selection_basis in targets:
        t = float(metas["standard"]["results"][lang]["threshold"])
        f1_target = doc_f1(y, std[lang], t)
        difference = f1_en - f1_target
        eligible = np.flatnonzero(difference >= 0.5 - 1e-12)
        case_stats.append({"lang": lang.upper(), "selection_basis": selection_basis,
                           "eligible_count": len(eligible), "test_docs": len(y),
                           "eligible_pct": round(len(eligible) / len(y) * 100, 3)})
        # All candidates, ordered by CELEX/test order.
        for ix in eligible:
            all_candidates.append({
                "lang": lang.upper(), "celex_id": str(ids[ix]),
                "row_index_0based": int(ix),
                "en_document_f1": round(float(f1_en[ix]), 6),
                "target_document_f1": round(float(f1_target[ix]), 6),
                "f1_difference_en_minus_target": round(float(difference[ix]), 6),
            })
        chosen = np.random.default_rng(seed).choice(eligible, size=min(20, len(eligible)), replace=False)
        for n_case, ix in enumerate(chosen, start=1):
            en_predict = pred_labels(std["en"][ix], float(metas["standard"]["results"]["en"]["threshold"]))
            target_predict = pred_labels(std[lang][ix], t)
            # Do not call the coded 'cause' before the expert has read the original texts.
            row = {
                "case_id": f"{lang.upper()}_{n_case:02d}",
                "lang": lang.upper(),
                "selection_basis": selection_basis,
                "celex_id": str(ids[ix]),
                "row_index_0based": int(ix),
                "true_label_indices": json.dumps(true_labels(y[ix]), ensure_ascii=False),
                "en_prediction_indices": json.dumps(en_predict, ensure_ascii=False),
                "target_prediction_indices": json.dumps(target_predict, ensure_ascii=False),
                "en_document_f1": round(float(f1_en[ix]), 6),
                "target_document_f1": round(float(f1_target[ix]), 6),
                "f1_difference_en_minus_target": round(float(difference[ix]), 6),
                "en_probabilities_21": json.dumps([round(float(v), 6) for v in std["en"][ix]], ensure_ascii=False),
                "target_probabilities_21": json.dumps([round(float(v), 6) for v in std[lang][ix]], ensure_ascii=False),
                "en_full_text": "",  # TODO: merge official EN parquet by CELEX, DO NOT invent text.
                "target_full_text": "",  # TODO: merge official FI / FR parquet by CELEX.
                "text_status": "NEEDS_ORIGINAL_TEXT_JOIN",
            }
            selected.append(row)
            for coder, output in (("D", coding_d), ("E", coding_e)):
                jr = {k: v for k, v in row.items() if k not in ["en_probabilities_21", "target_probabilities_21"]}
                jr["coder"] = coder
                jr.update({k: "" for k in CATEGORIES})
                jr["error_notes"] = ""
                output.append(jr)
    return case_stats, all_candidates, selected, coding_d, coding_e

## 5. 코드에 포함된 간단한 Kappa 유틸리티

In [ ]:
def kappa_for_binary(a, b):
    a = np.asarray(a, dtype=np.int8)
    b = np.asarray(b, dtype=np.int8)
    po = np.mean(a == b)
    p1_a = np.mean(a == 1)
    p1_b = np.mean(b == 1)
    pe = p1_a * p1_b + (1 - p1_a) * (1 - p1_b)
    kappa = float((po-pe)/(1-pe)) if not np.isclose(pe, 1) else float('nan')
    return float(po), kappa

def calculate_kappa(d_csv, e_csv, out_csv):
    """Call after D and E independently complete 0/1 columns for all cases."""
    with open(d_csv, encoding="utf-8-sig", newline="") as f:
        d = list(csv.DictReader(f))
    with open(e_csv, encoding="utf-8-sig", newline="") as f:
        e = list(csv.DictReader(f))
    dm, em = {r["case_id"]: r for r in d}, {r["case_id"]: r for r in e}
    if set(dm) != set(em):
        raise ValueError("D/E case_id sets differ")
    rows = []
    for name in CATEGORIES:
        av, bv = [], []
        for case_id in sorted(dm):
            x, z = dm[case_id][name].strip(), em[case_id][name].strip()
            if x not in ("0", "1") or z not in ("0", "1"):
                raise ValueError(f"Missing/invalid 0/1 code for {name} at {case_id}")
            av.append(int(x)); bv.append(int(z))
        agreement, kappa = kappa_for_binary(av, bv)
        rows.append({"category": name, "n_cases": len(av),
                     "observed_agreement_pct": round(agreement * 100, 4),
                     "cohens_kappa": round(kappa, 6) if np.isfinite(kappa) else "undefined",
                     "d_positives": sum(av), "e_positives": sum(bv)})
    write_csv(out_csv, rows, list(rows[0]))
    return rows

## 6. 전체 분석 실행 함수

CLI용 `main()`을 Notebook 환경에서 직접 호출할 수 있는 `run_analysis(...)`로 바꾼 부분입니다. 내부 분석 로직은 동일합니다.

In [ ]:
def run_analysis(data_dir="/mnt/data", out_dir="/mnt/data/F_analysis",
                 n_bootstrap=10000, seed=42, kappa_only=False):
    args = argparse.Namespace(
        data_dir=str(data_dir), out_dir=str(out_dir),
        n_bootstrap=int(n_bootstrap), seed=int(seed),
        kappa_only=bool(kappa_only),
    )
    out = Path(args.out_dir); out.mkdir(parents=True, exist_ok=True)
    if args.kappa_only:
        print(calculate_kappa(out / "coding_D.csv", out / "coding_E.csv", out / "cohens_kappa.csv"))
        return
    data = Path(args.data_dir)
    y_path = data / "en_test_Y.npy"
    if not y_path.exists():
        y_path = data / "en_test_Y.npy.bin"
    y = np.load(y_path, allow_pickle=False)
    ids = np.asarray(read_json(data / "en_test_celex.json"), dtype=str)
    assert y.shape == (5000, 21)
    assert len(set(ids)) == 5000
    preds, metas = {}, {}
    for cond in ("standard", "diagnostic", "direct"):
        preds[cond] = np.load(data / f"{cond}_test.npz", allow_pickle=False)
        metas[cond] = read_json(data / f"{cond}_test.json")
        assert np.array_equal(preds[cond]["ids"], ids), f"CELEX mismatch: {cond}"
        assert set(preds[cond].files) - {"ids"} == set(metas[cond]["results"])
    # Diagnostic probabilities must be identical to Standard, with only threshold changed.
    for lang in TARGET_LANGS:
        assert np.array_equal(preds["diagnostic"][lang], preds["standard"][lang])
    rows, n_models = bootstrap_rows(y, preds, metas, args.n_bootstrap, args.seed)
    write_csv(out / "bootstrap_95ci.csv", rows, list(rows[0]))
    stats, candidates, cases, coding_d, coding_e = make_flips(y, ids, preds["standard"], metas, rows, args.seed)
    write_csv(out / "flip_candidate_counts.csv", stats, list(stats[0]))
    write_csv(out / "flip_all_eligible.csv", candidates, list(candidates[0]))
    write_csv(out / "flip_selected_40_cases.csv", cases, list(cases[0]))
    write_csv(out / "coding_D.csv", coding_d, list(coding_d[0]))
    write_csv(out / "coding_E.csv", coding_e, list(coding_e[0]))
    manifest = {"inputs": ["standard_test.npz/json", "diagnostic_test.npz/json",
                            "direct_test.npz/json", "en_test_Y.npy.bin", "en_test_celex.json"],
                "n_docs": 5000, "n_labels": 21, "n_models_verified": n_models,
                "n_bootstrap": args.n_bootstrap, "seed": args.seed,
                "resampling_unit": "CELEX document; paired across all models",
                "bootstrap_ci": "95% percentile", "bootstrap_output_unit": "percentage points",
                "doc_f1_zero_division": 0,
                "flip_rule": "doc_F1(en_standard)-doc_F1(target_standard) >= 0.5",
                "selected_n": len(cases), "flip_candidates": stats,
                "original_texts_joined": False,
                "note": "Original EN and target document texts must be joined by CELEX before D/E qualitative coding. Neither annotation nor Cohen's Kappa is fabricated."}
    (out / "analysis_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({"n_models_verified": n_models, "flip_candidates": stats,
                      "bootstrap_sample": rows[:4]}, ensure_ascii=False, indent=2))

## 7. 경로 설정 및 실행

`RUN_ANALYSIS = True`로 바꾸면 실행됩니다. **정답·예측 파일이 모두 데이터 폴더에 있는지**, 그리고 출력 폴더에 D/E가 이미 작성한 코딩 파일이 없는지 먼저 확인하세요.

In [ ]:
DATA_DIR = Path("/mnt/data")      #  A 정답 + B 예측 NPZ/JSON 위치
OUTPUT_DIR = Path("/mnt/data/F_analysis_notebook")
N_BOOTSTRAP = 10000
SEED = 42
RUN_ANALYSIS = False          # 파일 준비 후 True로 변경해 실행

if RUN_ANALYSIS:
    run_analysis(
        data_dir=DATA_DIR,
        out_dir=OUTPUT_DIR,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED,
    )
else:
    print("준비 완료: 입력 파일·출력 경로 확인 후 RUN_ANALYSIS=True로 변경하세요.")